### Biblioteki

In [2]:
import numpy as np
import pandas as pd
import plotly as plt
import dash
import json

### Import

In [3]:
AK_SCEZKA = r"Dane prywatne/aw-bucket-export_aw-watcher-window_A14 (1).json"
AK_AFK_SCEZKA = r"Dane prywatne/aw-bucket-export_aw-watcher-afk_A14 (1).json"
with open(AK_AFK_SCEZKA, 'r', encoding='utf-8') as f: #Scieżka
    raw_data = json.load(f)

bucket_name = "aw-watcher-afk_A14" #Nazwa bucketa w której jest wewnętrzna nazwa urządzenia
events_list = raw_data['buckets'][bucket_name]['events']

df1 = pd.json_normalize(events_list)
# df1['timestamp'] = pd.to_datetime(df1['timestamp'])

with open(AK_SCEZKA, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

bucket_name = "aw-watcher-window_A14"
events_list = raw_data['buckets'][bucket_name]['events']

df2 = pd.json_normalize(events_list)


### Ultimate APP + AFK

In [7]:
import pandas as pd
import numpy as np

# Zakładam, że df1 (AFK) i df2 (Window) są już wczytane
df_afk = df1.copy()
df_window = df2.copy()

# --- KROK 1: Konwersja i przygotowanie ---
# Konwersja na datetime (UTC) - to jest "normalny, wydajny format"
df_afk['timestamp'] = pd.to_datetime(df_afk['timestamp'], format='ISO8601', utc=True)
df_window['timestamp'] = pd.to_datetime(df_window['timestamp'], format='ISO8601', utc=True)

# Opcjonalnie: Konwersja na strefę czasową Polski (dla czytelności)
# df_afk['timestamp'] = df_afk['timestamp'].dt.tz_convert('Europe/Warsaw')
# df_window['timestamp'] = df_window['timestamp'].dt.tz_convert('Europe/Warsaw')

# Obliczenie czasu zakończenia (Timedelta jest bardzo wydajna)
df_afk['end_time'] = df_afk['timestamp'] + pd.to_timedelta(df_afk['duration'], unit='s')
df_window['end_time'] = df_window['timestamp'] + pd.to_timedelta(df_window['duration'], unit='s')

# --- KROK 2: Oś czasu (Master Timeline) ---
all_timestamps = pd.concat([
    df_window['timestamp'], 
    df_window['end_time'], 
    df_afk['timestamp'], 
    df_afk['end_time']
]).unique()

# Sortujemy i tworzymy DataFrame (tylko RAZ, usuwając duplikat z Twojego kodu)
df_master = pd.DataFrame({'timestamp': all_timestamps})
df_master = df_master.sort_values('timestamp').reset_index(drop=True)

# --- KROK 3: Łączenie (Merge Asof) ---
# Przygotowanie nazw kolumn
df_window = df_window.rename(columns={'data.app': 'app', 'data.title': 'title'})
df_afk = df_afk.rename(columns={'data.status': 'status'})

# Dołączanie informacji o Oknie
df_master = pd.merge_asof(
    df_master, 
    df_window[['timestamp', 'end_time', 'app', 'title']].sort_values("timestamp"), 
    on='timestamp', 
    direction='backward',
    suffixes=('', '_win')
)

# Dołączanie informacji o AFK
df_master = pd.merge_asof(
    df_master, 
    df_afk[['timestamp', 'end_time', 'status']].sort_values("timestamp"), 
    on='timestamp', 
    direction='backward',
    suffixes=('', '_afk')
)

# --- KROK 4: Obliczenie Duration (NAJWAŻNIEJSZA ZMIANA) ---
# Liczymy czas trwania TERAZ, na pełnej osi czasu, włączając przerwy
# Dzięki temu "dziura" w czasie zostanie policzona jako osobny (później odrzucony) wiersz
df_master['duration'] = df_master['timestamp'].diff().shift(-1).dt.total_seconds()

# --- KROK 5: Filtrowanie aktywności ---
is_window_active = df_master['timestamp'] < df_master['end_time']
is_afk_active = df_master['timestamp'] < df_master['end_time_afk']

# Zostawiamy tylko momenty, gdzie okno było faktycznie otwarte
df_final = df_master[is_window_active].copy()

# Uzupełnianie statusu AFK
# Jeśli brak danych o AFK, a okno działa, zakładamy 'not-afk'
# Używamy np.where dla wydajności wektorowej (szybsze niż .loc przy dużych danych)
df_final['status'] = np.where(
    df_final['timestamp'] < df_final['end_time_afk'], 
    df_final['status'], 
    'not-afk'
)

# --- KROK 6: Czyszczenie końcowe ---
# Usuwamy wiersze z zerowym czasem lub błędami
df_final = df_final[df_final['duration'] > 0]

# Wybór kolumn
df = df_final[['timestamp', 'duration', 'app', 'title', 'status']].reset_index(drop=True)

# --- SPRAWDZENIE FORMATU ---
print("Typy danych:")
print(df.dtypes)
print("\nPrzykładowe dane:")
print(df.head())

Typy danych:
timestamp    datetime64[ns, UTC]
duration                 float64
app                       object
title                     object
status                    object
dtype: object

Przykładowe dane:
                         timestamp  duration           app  \
0 2025-12-13 21:59:13.306000+00:00     7.122  explorer.exe   
1 2025-12-13 21:59:21.440000+00:00    34.438    chrome.exe   
2 2025-12-13 21:59:57.900000+00:00    43.490    chrome.exe   
3 2025-12-13 22:00:42.399000+00:00     5.056  explorer.exe   
4 2025-12-13 22:00:49.478000+00:00     6.095  explorer.exe   

                                               title   status  
0                                            Pobrane  not-afk  
1  Downloads | ActivityWatch - Open-source time t...  not-afk  
2                      ActivityWatch - Google Chrome  not-afk  
3                                            Pobrane  not-afk  
4                                          Dokumenty  not-afk  


In [8]:
df

,timestamp,duration,app,title,status
0,2025-12-13 21:59:13.306000+00:00,7.122,explorer.exe,Pobrane,not-afk
1,2025-12-13 21:59:21.440000+00:00,34.438,chrome.exe,Downloads | ActivityWatch - Open-source time t...,not-afk
2,2025-12-13 21:59:57.900000+00:00,43.490,chrome.exe,ActivityWatch - Google Chrome,not-afk
3,2025-12-13 22:00:42.399000+00:00,5.056,explorer.exe,Pobrane,not-afk
4,2025-12-13 22:00:49.478000+00:00,6.095,explorer.exe,Dokumenty,not-afk
...,...,...,...,...,...
12613,2026-01-09 19:30:34.668000+00:00,2.018,Code.exe,● AK-komp-0901aw-buckets-export.json - TWD_Das...,not-afk
12614,2026-01-09 19:30:37.699000+00:00,3.045,Code.exe,● start.ipynb - TWD_Dashboard - Visual Studio ...,not-afk
12615,2026-01-09 19:30:43.775000+00:00,3.037,Discord.exe,Friends - Discord,not-afk
12616,2026-01-09 19:30:47.821000+00:00,11.128,Discord.exe,@Kosiaromocny - Discord,not-afk


In [ ]:
wynik = df.groupby('app')['duration'].sum()
wynik.sort_values(ascending=False)/3600

In [15]:
wynik = df[df['status'] == 'afk'].groupby('app')['duration'].sum()
wynik.sort_values(ascending=False)/3600

app
chrome.exe      16.913106
explorer.exe     1.175509
aces.exe         0.643376
Code.exe         0.456687
ms-teams.exe     0.396052
rstudio.exe      0.346037
Discord.exe      0.329567
MATLAB.exe       0.194117
Name: duration, dtype: float64

In [17]:
print(df2.groupby("data.app")["duration"].sum().sort_values(ascending=False))/3600

data.app
chrome.exe                  342343.669
MATLAB.exe                   32554.079
aces.exe                     23963.472
Code.exe                     22618.819
explorer.exe                 14430.384
ms-teams.exe                  9355.217
Discord.exe                   7101.400
eclipse.exe                   6958.206
rstudio.exe                   6059.102
javaw.exe                     4651.511
notepad.exe                   2078.187
Acrobat.exe                    649.846
cmd.exe                        572.101
Photos.exe                     561.666
ShellExperienceHost.exe        384.840
Minecraft.exe                  171.879
launcher.exe                   157.678
ScreenClippingHost.exe         102.318
POWERPNT.EXE                    56.867
msedge.exe                      35.389
SearchApp.exe                   32.344
ApplicationFrameHost.exe        27.243
msdt.exe                        24.664
Taskmgr.exe                     19.675
EXCEL.EXE                       16.187
WinRAR.exe      

TypeError: unsupported operand type(s) for /: 'NoneType' and 'int'

In [18]:
df.to_json('Dane/PC-SUM-A-09.01.2026.json', orient='records', indent=4, date_format='iso',force_ascii=False)